Step 1: Load Dataset

In [1]:
import pandas as pd
import numpy as np
path = r"E:\Mayuri\DataSets\historical_data.csv"
#Load data
df = pd.read_csv(path)
#preview data
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,22:24:17,23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,21:49:25,22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,20:39:28,21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,21:21:45,22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,02:40:36,03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


Step 2: Basic Data Quality Check

In [2]:
df.shape


(197428, 16)

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 16 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   market_id                                     196441 non-null  float64
 1   created_at                                    197428 non-null  str    
 2   actual_delivery_time                          197421 non-null  str    
 3   store_id                                      197428 non-null  int64  
 4   store_primary_category                        192668 non-null  str    
 5   order_protocol                                196433 non-null  float64
 6   total_items                                   197428 non-null  int64  
 7   subtotal                                      197428 non-null  int64  
 8   num_distinct_items                            197428 non-null  int64  
 9   min_item_price                                197428 non-nu

In [4]:
df.describe()


,market_id,store_id,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
count,196441.000000,197428.000000,196433.000000,197428.000000,197428.000000,197428.000000,197428.000000,197428.000000,181166.000000,181166.000000,181166.000000,197428.000000,196902.000000
mean,2.978706,3530.510272,2.882352,3.196391,2682.331402,2.670791,686.218470,1159.588630,44.808093,41.739747,58.050065,308.560179,545.358935
std,1.524867,2053.496711,1.503771,2.666546,1823.093688,1.630255,522.038648,558.411377,34.526783,32.145733,52.661830,90.139653,219.352902
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,-86.000000,0.000000,-4.000000,-5.000000,-6.000000,0.000000,0.000000
25%,2.000000,1686.000000,1.000000,2.000000,1400.000000,1.000000,299.000000,800.000000,17.000000,15.000000,17.000000,251.000000,382.000000
50%,3.000000,3592.000000,3.000000,3.000000,2200.000000,2.000000,595.000000,1095.000000,37.000000,34.000000,41.000000,251.000000,544.000000
75%,4.000000,5299.000000,4.000000,4.000000,3395.000000,3.000000,949.000000,1395.000000,65.000000,62.000000,85.000000,446.000000,702.000000
max,6.000000,6987.000000,7.000000,411.000000,27100.000000,20.000000,14700.000000,14700.000000,171.000000,154.000000,285.000000,2715.000000,2088.000000


In [5]:
# Check missing values
df.isnull().sum()


market_id                                         987
created_at                                          0
actual_delivery_time                                7
store_id                                            0
store_primary_category                           4760
order_protocol                                    995
total_items                                         0
subtotal                                            0
num_distinct_items                                  0
min_item_price                                      0
max_item_price                                      0
total_onshift_dashers                           16262
total_busy_dashers                              16262
total_outstanding_orders                        16262
estimated_order_place_duration                      0
estimated_store_to_consumer_driving_duration      526
dtype: int64

Step 3: Create Target Variable

In [6]:
# Convert timestamps
df['created_at'] = pd.to_datetime(df['created_at'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])

# Create target variable (in seconds)
df['delivery_duration'] = (
    df['actual_delivery_time'] - df['created_at']
).dt.total_seconds()

# Remove rows with missing target
df = df.dropna(subset=['delivery_duration'])

df['delivery_duration'].describe()


C:\Users\archa\AppData\Local\Temp\ipykernel_9128\926560639.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at'] = pd.to_datetime(df['created_at'])
C:\Users\archa\AppData\Local\Temp\ipykernel_9128\926560639.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])


count    197421.000000
mean        253.074794
std       14795.404986
min      -85869.000000
25%        2048.000000
50%        2622.000000
75%        3347.000000
max       73282.000000
Name: delivery_duration, dtype: float64

Step 4: Remove Outliers

In [7]:
# Keep deliveries between 2 minutes and 2 hours
df = df[
    (df['delivery_duration'] > 120) &
    (df['delivery_duration'] < 7200)
]


Step 5: Feature Engineering

In [8]:
# Convert Created_at to datetime
df['created_at'] = pd.to_datetime(df['created_at'])

#Time Feature
df['hour'] = df['created_at'].dt.hour
df['day_of_week'] = df['created_at'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)


In [16]:
# Order Feature
df['avg_item_price'] = df['subtotal'] / df['total_items'].replace(0, np.nan)
df['price_range'] = df['max_item_price'] - df['min_item_price']


In [10]:
df['busy_ratio'] = (
    df['total_busy_dashers'] /
    df['total_onshift_dashers'].replace(0, np.nan)
)

# Handle infinite / missing values
df['busy_ratio'] = df['busy_ratio'].replace([np.inf, -np.inf], np.nan)
df['busy_ratio'] = df['busy_ratio'].fillna(df['busy_ratio'].median())


In [17]:
##Handle Missing Values
# Fill numeric missing values with median
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])


C:\Users\archa\AppData\Local\Temp\ipykernel_9128\1896828866.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


Step 6: feature Selection

In [38]:
features =['market_id', 'created_at', 'actual_delivery_time', 'store_id',
       'store_primary_category', 'order_protocol', 'total_items', 'subtotal',
       'num_distinct_items', 'min_item_price', 'max_item_price',
       'total_onshift_dashers', 'total_busy_dashers',
       'total_outstanding_orders', 'estimated_order_place_duration',
       'estimated_store_to_consumer_driving_duration', 'delivery_duration',
       'hour', 'day_of_week', 'is_weekend', 'avg_item_price', 'price_range',
       'busy_ratio']

In [21]:
print(df.columns)

Index(['market_id', 'created_at', 'actual_delivery_time', 'store_id',
       'store_primary_category', 'order_protocol', 'total_items', 'subtotal',
       'num_distinct_items', 'min_item_price', 'max_item_price',
       'total_onshift_dashers', 'total_busy_dashers',
       'total_outstanding_orders', 'estimated_order_place_duration',
       'estimated_store_to_consumer_driving_duration', 'delivery_duration',
       'hour', 'day_of_week', 'is_weekend', 'avg_item_price', 'price_range',
       'busy_ratio'],
      dtype='str')


In [22]:
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,...,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration,delivery_duration,hour,day_of_week,is_weekend,avg_item_price,price_range,busy_ratio
0,1.0,2026-02-17 22:24:17,2026-02-17 23:27:16,1845,american,1.0,4,3441,4,557,...,21.0,446,861.0,3779.0,22,1,0,860.25,682,0.424242
1,2.0,2026-02-17 21:49:25,2026-02-17 22:56:29,5477,mexican,2.0,1,1900,1,1400,...,2.0,446,690.0,4024.0,21,1,0,1900.00,0,2.000000
2,3.0,2026-02-17 20:39:28,2026-02-17 21:09:09,5477,american,1.0,1,1900,1,1900,...,0.0,446,690.0,1781.0,20,1,0,1900.00,0,0.000000
3,3.0,2026-02-17 21:21:45,2026-02-17 22:13:00,5477,american,1.0,6,6900,5,600,...,2.0,446,289.0,3075.0,21,1,0,1150.00,1200,1.000000
4,3.0,2026-02-17 02:40:36,2026-02-17 03:20:26,5477,american,1.0,3,3900,3,1100,...,9.0,446,650.0,2390.0,2,1,0,1300.00,500,1.000000


In [39]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

Step 7: Train-Test Split

In [40]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (152387, 17)
Test: (38097, 17)


 Step 8: Preprocessing

In [41]:
categorical_features = [
    'market_id',
    'store_primary_category',
    'order_protocol'
]

numerical_features = [
    col for col in features if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


step 9: Linear Regression Model

In [32]:
linear_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)


In [33]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results")
print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))
print("R2 Score:", round(r2,3))


Linear Regression Results
MAE: 656.16
RMSE: 861.33
R2 Score: 0.277
